In [2]:
%pip install unsloth

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
%pip install --upgrade unsloth-zoo
%pip install --upgrade unsloth

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
%pip install ipywidgets -U

In [4]:
%pip install tqdm -U

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [ ]:
!nvidia-smi

Thu Oct  2 11:47:26 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:8C:00.0 Off |                    0 |
| N/A   32C    P0              76W / 500W |      9MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

### Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch

# fourbit_models = [
#     "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
#     "unsloth/Qwen3-4B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-8B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-14B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-32B-unsloth-bnb-4bit",

#     # 4bit dynamic quants for superior accuracy and low memory use
#     "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
#     "unsloth/Phi-4",
#     "unsloth/Llama-3.1-8B",
#     "unsloth/Llama-3.2-3B",
#     "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
# ] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/Qwen3-0.6B",
    model_name = "models/qwen3_0.6b-reviews-fine-tune-v4",
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    # load_in_4bit = False,     # 4bit uses much less memory
    # load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = True, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-10-02 14:25:15.609965: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-02 14:25:16.400177: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2025.9.10: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.325 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.


In [ ]:
model.push_to_hub("JosephThePatrician/qwen3_0.6b-reviews-fine-tune-v4", tokenizer, token = "token")
tokenizer.push_to_hub("JosephThePatrician/qwen3_0.6b-reviews-fine-tune-v4", token = "token")

In [3]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",],
#     lora_alpha = 64,  # Best to choose alpha = rank or rank*2
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,   # We support rank stabilized LoRA
#     loftq_config = None,  # And LoftQ
# )

In [4]:
SYSTEM_PROMPT = """\
Проанализируй отзыв клиента Газпромбанка (ГПБ) и определи:
1. Упоминаемые тем ("topic") из списка допустимых тем;
2. Тональность ("sentiment") для каждой темы: positive/negative/neutral.

ПРАВИЛА:
- Тональность: `neutral` указывается тогда, когда тема упомянута как факт, без эмоциональной окраски.
- Не выдумывай темы: Если в отзыве нет явного упоминания продукта или услуги, не включай его.
- Если темы нет: Если невозможно определить ни одну тему, верни пустой массив [].
- Темы: Используй ТОЛЬКО следующий список тем и подтем. Не добавляй новые темы.
- Уникальность тем: Одна тема может встречаться только один раз.

ДОПУСТИМЫЕ ТЕМЫ:
- Офисное обслуживание (обслуживание в отделениях банка)
- Дистанционное обслуживание (звонки, чаты, онлайн-консультации и подобное)
- Банкоматы
- Курьерская доставка карт
- Обмен валют
- Дебетовые карты (включая подтемы: Денежные переводы, Карта UnionPay, Умная дебетовая карта «Мир», Премиальная карта Mir Supreme)
- Кредитные карты (включая подтемы: Кредитная карта 180 дней Премиум)
- Кредиты (включая подтемы: Кредит наличными, Кредит наличными под залог недвижимости, Кредит под залог автомобиля)
- Рефинансирование/Реструктуризация (включая подтемы: Рефинансирование кредитов, Реструктуризация кредитов, Рефинансирование ипотеки, Реструктуризация ипотеки)
- Автокредиты
- Ипотека
- Страховые и сервисные продукты
- Вклады (включая подтемы: Вклад «Копить», Вклад «В Плюсе», Вклад «Новые деньги»)
- Накопительные счета (включая подтемы: Накопительный счёт «Ежедневная выгода», Накопительный счёт «Ежедневный процент», Накопительный счёт «Премиум»)
- Акции и бонусы (включая подтемы: Газпром Бонус, Газпромбанк Привилегии, Кэшбэк, Акции, Программы лояльности)
- Газпромбанк Премиум (включая подтемы: Персональный менеджер, Консьерж-сервис, Премиальное обслуживание)
- Мобильное приложение
- Другие услуги банка (включая подтемы: Газпромбанк Travel (покупка авиабилетов/отелей), Gazprom Pay (оплата телефоном), GorodPay (оплата общественного транспорта), Инвестиционные продукты, Брокерские услуги, Депозитарные услуги, Аренда сейфовых ячеек)

Примеры:
Отзыв: "В отделении грубо обслужили, но мобильное приложение удобное"
[
{"topic": "Офисное обслуживание", "sentiment": "negative"},
{"topic": "Мобильное приложение", "sentiment": "positive"}
]

Отзыв: "Курьер не пришёл на встречу. По телефону не смогли помочь."
[
{"topic": "Курьерская доставка карт", "sentiment": "negative"},
{"topic": "Дистанционное обслуживание", "sentiment": "negative"}
]

Отзыв: "Оформил Премиальную карту Mir Supreme через приложение"
[
{"topic": "Премиальная карта Mir Supreme", "sentiment": "neutral"},
{"topic": "Дебетовые карты", "sentiment": "neutral"},
{"topic": "Газпромбанк Премиум", "sentiment": "neutral"},
{"topic": "Мобильное приложение", "sentiment": "neutral"}
]

Отзыв: "Пользуюсь Газпромбанк Travel для бронирования отелей и Gazprom Pay для оплаты"
[
{"topic": "Другие услуги банка", "sentiment": "neutral"},
{"topic": "Газпромбанк Travel", "sentiment": "neutral"},
{"topic": "Gazprom Pay", "sentiment": "neutral"}
]

Проанализируй следующий отзыв:
"""

In [5]:
import json
import pandas as pd
import numpy as np

import glob
import os

In [6]:
# df = pd.read_csv("drive/MyDrive/lct/data_latest.csv")

# df

In [7]:
# Сдеалть разделение на train/test по дате

from datasets import Dataset

topics_sentiments_json = "dataset_v1.json"
# original_reviews_csv = "mount/data/data_latest.csv"

with open(topics_sentiments_json) as f:
    topics_sentiments_full = json.load(f)
    
# topics_sentiments_pair = {
#     int(topics_sentiments_full[i]["id"]) : topics_sentiments_full[i]["topic_sentiment_pairs"] 
#     for i in range(len(topics_sentiments_full))
# }

# original_reviews_df = pd.read_csv(original_reviews_csv)


user_prompts = []
assistant_answers = []

for i in range(len(topics_sentiments_full)):
    # original_review_series = original_reviews_df.iloc[i]
    # review_id = original_review_series["review_id"]

    # user_prompt = original_review_series["review_text"]
    
    user_prompt = topics_sentiments_full[i]["review_text"]

    assistant_answer = str(topics_sentiments_full[i]["topic_sentiment_pairs"])
    
    user_prompts.append(user_prompt + " /no_think")
    assistant_answers.append(assistant_answer)
    

dataset_dict = {"user_prompt" : user_prompts, "assistant_answer" : assistant_answers}

dataset = Dataset.from_dict(dataset_dict)

In [8]:
dataset = dataset.train_test_split(test_size=0.1, shuffle=True)

dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [9]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["user_prompt"]},
            {"role": "assistant", "content": example["assistant_answer"]}
        ]
    }

dataset_train = dataset_train.map(
    convert_to_chatml
)

dataset_test = dataset_test.map(
    convert_to_chatml
)

Map: 100%|██████████| 2768/2768 [00:00<00:00, 7526.48 examples/s] 


In [10]:
# from unsloth.chat_templates import standardize_sharegpt

# dataset_train = standardize_sharegpt(dataset_train)
# dataset_test = standardize_sharegpt(dataset_test)

# dataset_train = tokenizer.apply_chat_template(
#     dataset_train["conversations"],
#     tokenize = False,
# )

# dataset_test = tokenizer.apply_chat_template(
#     dataset_test["conversations"],
#     tokenize = False,
# )

In [11]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False) for convo in convos]
    return { "text" : texts, }

dataset_train = dataset_train.map(formatting_prompts_func, batched = True)

dataset_test = dataset_test.map(formatting_prompts_func, batched = True)

Map: 100%|██████████| 2768/2768 [00:00<00:00, 4828.06 examples/s]


In [12]:
dataset_train["text"][10]

'<|im_start|>system\nПроанализируй отзыв клиента Газпромбанка (ГПБ) и определи:\n1. Упоминаемые тем ("topic") из списка допустимых тем;\n2. Тональность ("sentiment") для каждой темы: positive/negative/neutral.\n\nПРАВИЛА:\n- Тональность: `neutral` указывается тогда, когда тема упомянута как факт, без эмоциональной окраски.\n- Не выдумывай темы: Если в отзыве нет явного упоминания продукта или услуги, не включай его.\n- Если темы нет: Если невозможно определить ни одну тему, верни пустой массив [].\n- Темы: Используй ТОЛЬКО следующий список тем и подтем. Не добавляй новые темы.\n- Уникальность тем: Одна тема может встречаться только один раз.\n\nДОПУСТИМЫЕ ТЕМЫ:\n- Офисное обслуживание (обслуживание в отделениях банка)\n- Дистанционное обслуживание (звонки, чаты, онлайн-консультации и подобное)\n- Банкоматы\n- Курьерская доставка карт\n- Обмен валют\n- Дебетовые карты (включая подтемы: Денежные переводы, Карта UnionPay, Умная дебетовая карта «Мир», Премиальная карта Mir Supreme)\n- Кред

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [13]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test, # Can set up evaluation!
    args = SFTConfig(
        output_dir="qwen3_0.6b-reviews-fine-tune-v4",
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 10,
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        dataset_num_proc=0,
        seed = 3407,
        report_to = "none", # Use this for WandB etc
        do_eval=True,
        eval_strategy="steps",
        eval_steps=0.2,
    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 2768/2768 [00:01<00:00, 1502.96 examples/s]


In [14]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>system\n",
    response_part = "<|im_start|>assistant\n",
    num_proc=0,
)

Map: 100%|██████████| 2768/2768 [00:02<00:00, 1004.87 examples/s]


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,908 | Num Epochs = 2 | Total steps = 1,558
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 596,049,920 of 596,049,920 (100.00% trained)
  0%|          | 0/1558 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/to

Unsloth: Will smartly offload gradients to save VRAM!


  0%|          | 5/1558 [01:31<4:28:09, 10.36s/it] 

{'loss': 0.4853, 'grad_norm': 21.125, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.01}


  1%|          | 10/1558 [01:51<2:08:15,  4.97s/it]

...

{'loss': 0.0525, 'grad_norm': 2.015625, 'learning_rate': 7.351421188630492e-06, 'epoch': 1.27}


 64%|██████▍   | 995/1558 [1:20:20<41:42,  4.44s/it]

{'loss': 0.0555, 'grad_norm': 2.140625, 'learning_rate': 7.286821705426357e-06, 'epoch': 1.28}


 64%|██████▍   | 1000/1558 [1:20:41<38:48,  4.17s/it]

{'loss': 0.0518, 'grad_norm': 1.6953125, 'learning_rate': 7.222222222222223e-06, 'epoch': 1.28}


In [ ]:
model.save_pretrained("models/qwen3_0.6b-reviews-fine-tune-v4")  # Local saving
tokenizer.save_pretrained("models/qwen3_0.6b-reviews-fine-tune-v4")

In [ ]:
# import time

# time.sleep(10)

model.push_to_hub("JosephThePatrician/qwen3_0.6b-reviews-fine-tune-v3", token = "token")
tokenizer.push_to_hub("JosephThePatrician/qwen3_0.6b-reviews-fine-tune-v3", token = "token")

In [ ]:
# model.push_to_hub_merged("JosephThePatrician/qwen3_0.6b-reviews-fine-tune-v2", tokenizer, save_method = "merged_16bit", token = "token")